# **DUKA ELECTRONICS SALES EXPLORATION**

## **Excecutive Summary**

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(2026)

n = 1000

products = {
    "Phone": 15000,
    "Laptop": 65000,
    "Headphones": 2500,
    "Tablet": 28000,
    "Charger": 1200
}

sales = pd.DataFrame({
    "order_id": range(1001, 1001 + n),
    "order_date": pd.to_datetime("2026-01-01")
                  + pd.to_timedelta(rng.integers(0, 180, n), unit="D"),
    "branch": rng.choice(
        ["Nairobi", "Mombasa", "Kisumu", "Nakuru", "Eldoret"],
        n,
        p=[0.4, 0.2, 0.15, 0.15, 0.1]
    ),
    "product": rng.choice(list(products), n),
    "quantity": rng.integers(1, 6, n),
    "channel": rng.choice(
        ["In-store", "Online", "Phone order"],
        n,
        p=[0.55, 0.35, 0.10]
    ),
})

sales["unit_price"] = sales["product"].map(products)

sales.to_csv("duka_sales.csv", index=False)

print("duka_sales.csv created successfully")

duka_sales.csv created successfully


In [ ]:
sales = pd.read_csv(
    "duka_sales.csv",
    parse_dates=["order_date"]
)

### **Task one: Dataset Inspection**
This section performs the professional "first five minutes" inspection of the dataset to understand its structure, data types, completeness, date range, and key business categories before analysis begins.

In [ ]:
sales.head()
sales.tail()

sales.shape

sales.info()

sales.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB


,order_id,order_date,quantity,unit_price
count,1000.000000,1000,1000.000000,1000.000000
mean,1500.500000,2026-03-30 08:44:09.600000,3.012000,23918.300000
min,1001.000000,2026-01-01 00:00:00,1.000000,1200.000000
25%,1250.750000,2026-02-14 00:00:00,2.000000,2500.000000
50%,1500.500000,2026-03-29 00:00:00,3.000000,15000.000000
75%,1750.250000,2026-05-14 00:00:00,4.000000,28000.000000
max,2000.000000,2026-06-29 00:00:00,5.000000,65000.000000
std,288.819436,NaN,1.416991,23908.790546


In [ ]:
sales.nunique()

sales["branch"].value_counts(
    normalize=True
)

sales["channel"].value_counts(
    normalize=True
)

sales["order_date"].min()

sales["order_date"].max()

Timestamp('2026-06-29 00:00:00')

### Inspection Summary

The dataset loaded successfully and contains 1,000 transaction records. Data types appear correct, including the order_date column which was parsed as a datetime field. The dataset covers the January–June 2026 period and includes sales data from five branches, multiple products, and three sales channels. No significant quality issues were identified during the initial inspection.

### **Task 2: Feature Engineering**
This section creates additional columns that will support business analysis, including revenue, month, weekday, weekend indicators, and product price bands.

In [ ]:
sales["revenue"] = (
    sales["quantity"]
    *
    sales["unit_price"]
)
sales["month"] = (
    sales["order_date"]
    .dt.month_name()
)
sales["weekday"] = (
    sales["order_date"]
    .dt.day_name()
)
sales["is_weekend"] = (
    sales["weekday"]
    .isin(["Saturday","Sunday"])
)
sales["price_band"] = np.where(
    sales["unit_price"] >= 28000,
    "Premium",
    "Value"
)

### **Task 3: Branch Rankimg**

In [ ]:
branch_revenue = (
    sales.groupby("branch")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

branch_orders = (
    sales.groupby("branch")["order_id"]
    .count()
    .sort_values(ascending=False)
)

print("BRANCH REVENUE RANKING")
print(branch_revenue)

print("\nBRANCH ORDER RANKING")
print(branch_orders)



BRANCH REVENUE RANKING
branch
Nairobi    26106100
Mombasa    14003700
Nakuru     11970400
Kisumu     10868100
Eldoret     8844200
Name: revenue, dtype: int64

BRANCH ORDER RANKING
branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: order_id, dtype: int64


In [ ]:
branch_orders =(
sales.groupby("branch")[
    "order_id"
].count()
)

Nairobi ranks first in both revenue and orders. Some branches generate more revenue per order because they sell higher-priced products.

### **Task 4: Top products by revenue**

In [ ]:
sales.groupby(
    "product"
)["revenue"].sum()

,revenue
product,
Charger,642000
Headphones,1387500
Laptop,41535000
Phone,8880000
Tablet,19348000


### Findings
Branches were ranked by both revenue and order volume. The rankings may differ because revenue depends not only on the number of orders but also on the products sold and their price levels. Branches selling more Premium products can generate higher revenue even with fewer transactions. Therefore, both order count and revenue should be considered when evaluating branch performance.

### **Task 4: Product Performance Analysis**
This section identifies the top three products by revenue and compares them with the top three products by units sold.

In [ ]:
sales.groupby(
    "product"
)["quantity"].sum()

,quantity
product,
Charger,535
Headphones,555
Laptop,639
Phone,592
Tablet,691


The highest-revenue products are not necessarily the highest-selling products by quantity. Revenue is influenced by both volume and unit price, meaning expensive items such as laptops can generate substantial revenue even when fewer units are sold. Comparing both rankings provides a more complete view of product performance and profitability.

### **Task 5: Weekend vs weekday performance**
This section compares weekday and weekend sales performance in terms of order count, total revenue, and average order value.

In [ ]:
sales.groupby(
    "is_weekend"
)


In [ ]:
weekend_summary = sales.groupby("is_weekend")["revenue"].agg(
    order_count="count",
    total_revenue="sum",
    average_revenue="mean"
)

weekend_summary

,order_count,total_revenue,average_revenue
is_weekend,,,
False,723,51856200,71723.651452
True,277,19936300,71972.202166


### **Findings**
Weekday orders accounted for 723 transactions and generated total revenue of KES 51,856,200, while weekend orders accounted for 277 transactions and generated KES 19,936,300 in revenue. Although weekdays contributed most of the total revenue due to the higher number of orders, the average order value was slightly higher on weekends (KES 71,972.20) than on weekdays (KES 71,723.65). This suggests that customers spend marginally more per transaction during weekends, even though overall sales activity is concentrated on weekdays. The business may benefit from increasing weekend marketing efforts to attract more customers while maintaining the higher average order value observed during weekends.

### **Task 6: Best Month**
This section identifies the highest-performing and lowest-performing months based on revenue and measures the percentage difference between them.

In [ ]:
monthly_revenue = sales.groupby("month")["revenue"].sum()

monthly_revenue

,revenue
month,
April,13518300
February,11772000
January,11786600
June,10903400
March,12396800
May,11415400


In [ ]:
best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()

best_revenue = monthly_revenue.max()
worst_revenue = monthly_revenue.min()

print("Best Month:", best_month)
print("Worst Month:", worst_month)

Best Month: April
Worst Month: June


In [ ]:
percentage_difference = (
    (best_revenue - worst_revenue)
    / worst_revenue
) * 100

print(
    f"Percentage Difference: {percentage_difference:.2f}%"
)

Percentage Difference: 23.98%


### Findings
The month with the highest revenue was April, generating KES 11772000. The lowest-performing month was June, generating KES 10903400. The best month generated 23.98% more revenue than the worst month. This indicates substantial variation in monthly sales performance and may reflect differences in customer demand, product mix, or promotional activity.

### **Task 7: Premium online orders**
This section filters Premium Online orders from Nairobi and Mombasa and exports them for further analysis.

In [ ]:
premium_online = sales[
    (sales["price_band"] == "Premium")
    &
    (sales["channel"] == "Online")
    &
    (sales["branch"].isin(
        ["Nairobi", "Mombasa"]
    ))
]
order_count = len(
    premium_online
)

total_revenue = premium_online[
    "revenue"
].sum()

print("Order Count:", order_count)

print(
    f"Total Revenue: KES {total_revenue:,.0f}"
)

Order Count: 83
Total Revenue: KES 10,218,000


In [ ]:
import os

os.makedirs(
    "outputs",
    exist_ok=True
)
premium_online.to_csv(
    "outputs/premium_online_coast_capital.csv",
    index=False
)

### Findings
A total of 83 Premium Online orders were identified from Nairobi and Mombasa. These orders generated a combined revenue of KES 10,218,000. This customer segment represents high-value online purchases and could be targeted for additional digital marketing and loyalty campaigns.

## **Executive summary**
* Nairobi generated KES 26,106,100 and contributed 36.4% of total company revenue (Total Revenue: KES 71,792,500).
* April was the highest revenue month with KES 13,518,300 while June was the lowest revenue month with KES 10,903,400.
* Laptop generated the highest product revenue of KES 41,535,000.
* Weekday orders generated KES 51,856,200 from 723 transactions while weekend orders generated KES 19,936,300 from 277 transactions.
* Premium Online orders from Nairobi and Mombasa generated KES 10,218,000 across 83 orders.